# 🏠 Kaggle Starter Notebook: House Prices - Advanced Regression Techniques

Welcome to your first Kaggle Regression Notebook! In this project, we analyze residential home features from Ames, Iowa to predict final sale prices (`SalePrice`).

### 🎯 Project Objectives
1. **Exploratory Data Analysis (EDA)**: Understand target distribution, skewness, and key correlation factors.
2. **Data Cleaning & Preprocessing**: Handle missing values, encode categorical variables, and apply log transformations.
3. **Feature Engineering**: Construct domain-specific features like total square footage and total bathroom count.
4. **Machine Learning Pipelines**: Compare Ridge Regression, Random Forest, and LightGBM / Gradient Boosting using 5-Fold Cross Validation (RMSLE).
5. **Kaggle Submission Export**: Generate a formatted `submission.csv` ready for competition submission.

## 1. Setup & Environment Configuration

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore') # Suppress sklearn feature name warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False

pd.set_option('display.max_columns', 100)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("✅ Setup complete. Libraries loaded successfully!")

## 2. Data Acquisition & Inspection

In [ ]:
def create_synthetic_data(n_samples=600):
    np.random.seed(42)
    gr_liv = np.random.randint(800, 3500, size=n_samples)
    qual = np.random.randint(1, 10, size=n_samples)
    bsmt = np.random.randint(0, 2000, size=n_samples)
    year = np.random.randint(1950, 2021, size=n_samples)
    neighborhood = np.random.choice(['CollgCr', 'Veenker', 'Crawfor', 'NoRidge', 'Mitchel'], size=n_samples)
    
    price = 30000 + (gr_liv * 65) + (qual * 16000) + (bsmt * 45) + ((year - 1950) * 550) + np.random.normal(0, 12000, n_samples)
    price = np.maximum(price, 50000)
    
    df = pd.DataFrame({
        'Id': np.arange(1, n_samples + 1),
        'MSSubClass': np.random.choice([20, 60, 70, 120], size=n_samples),
        'Neighborhood': neighborhood,
        'OverallQual': qual,
        'YearBuilt': year,
        'TotalBsmtSF': bsmt,
        'GrLivArea': gr_liv,
        'FullBath': np.random.randint(1, 4, size=n_samples),
        'HalfBath': np.random.randint(0, 2, size=n_samples),
        'GarageCars': np.random.randint(0, 4, size=n_samples),
        'SalePrice': price
    })
    return df

kaggle_path = '../input/house-prices-advanced-regression-techniques/train.csv'
kaggle_test_path = '../input/house-prices-advanced-regression-techniques/test.csv'
local_path = 'train.csv'
local_test_path = 'test.csv'

if os.path.exists(kaggle_path):
    train_df = pd.read_csv(kaggle_path)
    test_df = pd.read_csv(kaggle_test_path)
    print("✅ Loaded Kaggle Competition Dataset from ../input/")
elif os.path.exists(local_path):
    train_df = pd.read_csv(local_path)
    test_df = pd.read_csv(local_test_path)
    print("✅ Loaded local train.csv and test.csv")
else:
    print("ℹ️ Dataset not attached. Running synthetic fallback...")
    synth = create_synthetic_data(600)
    train_df = synth.iloc[:400].copy()
    test_df = synth.iloc[400:].copy().drop(columns=['SalePrice'])

print(f"Training set shape: {train_df.shape}")
print(f"Testing set shape:  {test_df.shape}")
display(train_df.head())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(train_df['SalePrice'], kde=True, ax=axes[0], color='royalblue')
axes[0].set_title("Raw SalePrice Distribution (Right-Skewed)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("SalePrice ($)")

log_price = np.log1p(train_df['SalePrice'])
sns.histplot(log_price, kde=True, ax=axes[1], color='forestgreen')
axes[1].set_title("Log-Transformed log1p(SalePrice) (Normal Distribution)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("log1p(SalePrice)")

plt.tight_layout()
plt.show()

## 4. Feature Engineering & Preprocessing

In [ ]:
def engineer_features(df):
    df = df.copy()
    bsmt = df['TotalBsmtSF'] if 'TotalBsmtSF' in df.columns else 0
    gr_liv = df['GrLivArea'] if 'GrLivArea' in df.columns else 0
    df['TotalSF'] = bsmt + gr_liv
    
    full_bath = df['FullBath'] if 'FullBath' in df.columns else 0
    half_bath = df['HalfBath'] if 'HalfBath' in df.columns else 0
    df['TotalBath'] = full_bath + (0.5 * half_bath)
    
    if 'YearBuilt' in df.columns:
        df['HouseAge'] = 2026 - df['YearBuilt']
    return df

train_fe = engineer_features(train_df)
test_fe = engineer_features(test_df)

y_train_log = np.log1p(train_fe['SalePrice'])
X_train = train_fe.drop(columns=['Id', 'SalePrice'], errors='ignore')
X_test = test_fe.drop(columns=['Id', 'SalePrice'], errors='ignore')

num_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

print(f"Processed Features: Numerical={len(num_features)}, Categorical={len(cat_features)}")

## 5. Model Training & 5-Fold Cross Validation

In [ ]:
models = {
    'Ridge Regression': Ridge(alpha=10.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, random_state=42)
}
if HAS_LGBM:
    models['LightGBM'] = LGBMRegressor(n_estimators=100, learning_rate=0.05, random_state=42, verbose=-1)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    full_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    scores = cross_val_score(full_pipeline, X_train, y_train_log, cv=kf, scoring='neg_mean_squared_error')
    rmsle = np.sqrt(-scores)
    results[name] = rmsle
    print(f"📊 {name}: Mean RMSLE = {rmsle.mean():.4f} (Std = {rmsle.std():.4f})")

## 6. Final Predictions & Submission Generation

In [ ]:
best_model_name = min(results, key=lambda k: results[k].mean())
print(f"🏆 Selected Winning Model: {best_model_name}")

best_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', models[best_model_name])
])

best_pipeline.fit(X_train, y_train_log)
test_log_preds = best_pipeline.predict(X_test)
final_preds = np.expm1(test_log_preds)

sub_df = pd.DataFrame({
    'Id': test_df['Id'],
    'SalePrice': final_preds
})

sub_df.to_csv('submission.csv', index=False)
print(f"💾 Successfully generated 'submission.csv' with {len(sub_df)} rows!")
display(sub_df.head(10))